In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("DateFruit_Dataset.csv")
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [6]:
df.shape

(898, 35)

In [18]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [19]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [20]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)   

In [22]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [23]:
# ANN 

In [24]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [25]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)


X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [26]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [27]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [32]:
# Build Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

In [33]:
model = ANN()

# loss & optimizer

criteria = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

In [34]:
# Training the NN

epochs = 100
for epoch in range(epochs):
    model.train()
        
    running_loss = 0.0
        
    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() #params update
        
        optimizer.zero_grad()
        
        
        running_loss += loss
        
    train_loss = running_loss / len(train_loader)
    
    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.6763194799423218
epoch = 2/100, loss = 1.0962376594543457
epoch = 3/100, loss = 0.7162789106369019
epoch = 4/100, loss = 0.5381786823272705
epoch = 5/100, loss = 0.4402078092098236
epoch = 6/100, loss = 0.38428255915641785
epoch = 7/100, loss = 0.3331349492073059
epoch = 8/100, loss = 0.308640718460083
epoch = 9/100, loss = 0.27611714601516724
epoch = 10/100, loss = 0.2579263150691986
epoch = 11/100, loss = 0.237738236784935
epoch = 12/100, loss = 0.22801271080970764
epoch = 13/100, loss = 0.2204185128211975
epoch = 14/100, loss = 0.20399607717990875
epoch = 15/100, loss = 0.18609072268009186
epoch = 16/100, loss = 0.18397261202335358
epoch = 17/100, loss = 0.18214426934719086
epoch = 18/100, loss = 0.17108087241649628
epoch = 19/100, loss = 0.17316798865795135
epoch = 20/100, loss = 0.15842778980731964
epoch = 21/100, loss = 0.14933304488658905
epoch = 22/100, loss = 0.15545116364955902
epoch = 23/100, loss = 0.1550498604774475
epoch = 24/100, loss = 0.13977380

In [35]:
# Evaluate 
model.eval()
total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) 
        _, predicted = torch.max(outputs.data, 1)
        
        correct += (predicted == yb).sum().item() 
        total += yb.size(0) # actual samples in each batch

print("total vals:", total)
print("correct vals:", correct)
print(f"Accuracy = {100 * correct / total}%")

total vals: 180
correct vals: 168
Accuracy = 93.33333333333333%
